In [ ]:
import itertools
import pulp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from typing import Dict, List, Tuple, Any, Optional
from tqdm.notebook import tqdm
plt.rcParams['figure.dpi'] = 300

In [ ]:
class LineageTracker:
    def __init__(self, df: pd.DataFrame, cell_start: int = 1, lineage_start: int = 1):
        """
        LineageTracker manages cell lineage information across frames as a Pandas dataframe.

        cell_start:
            first cell_id to assign. This allows you to continue
            tracking across multiple, separate runs if desired.
        lineage_start:
            first lineage_id to assign. This allows you to continue
            tracking across multiple, separate runs if desired.
        """
        self.df = df.copy()

        # ensure linkage columns exist
        if 'parent_id' not in self.df.columns:
            self.df['parent_id'] = -1
        if 'parent_t' not in self.df.columns:
            self.df['parent_t'] = -1
        if 'lineage_id' not in self.df.columns:
            self.df['lineage_id'] = -1
        if 'cell_id' not in self.df.columns:
            self.df['cell_id'] = -1
        if 'generation' not in self.df.columns:
            self.df['generation'] = -1
        if 'division_event' not in self.df.columns:
            self.df['division_event'] = False
        if 'birth_event' not in self.df.columns:
            self.df['birth_event'] = False
        if 'death_event' not in self.df.columns:
            self.df['death_event'] = False

        # cell_counter hands out new cell IDs
        self.cell_counter = itertools.count(start=cell_start)

        # lineage_counter hands out new lineage IDs
        self.lineage_counter = itertools.count(start=lineage_start)

        # lineage_map records which (v,t,id) we've already given a lineage to
        # (v, t, id) -> lineage_id
        self.lineage_map = {}

    def initialize_lineages(self, v: int, t0: int = 0):
        """
        Initialize founder lineages for all cells in series v at frame t0.

        Every cell at (v, t0) becomes a new lineage root:
            cell_id = new unique ID
            lineage_id = new unique ID
            generation = 0
            parent_id = -1
            parent_t = -1
            division_event = False initially
            death_event = False initially
            birth_event = True

        Cells in other frames (t > t0) are left untouched here.
        """

        # subset rows at this series/time
        founders_mask = (self.df['v'] == v) & (self.df['t'] == t0)
        founders_idx = self.df.index[founders_mask].to_numpy()

        # nothing to init
        if founders_idx.size == 0:
            return

        # Assign a fresh lineage_id to each founder cell
        for idx in founders_idx:
            row = self.df.loc[idx]
            key = (int(row['v']), int(row['t']), int(row['id']))

            # if this founder already has a lineage_id (e.g. re-init), skip
            if key in self.lineage_map and self.df.at[idx, 'lineage_id'] >= 0:
                continue
            
            cell_id = next(self.cell_counter)
            lineage_id = next(self.lineage_counter)

            self.lineage_map[key] = lineage_id
            self.df.at[idx, 'cell_id'] = cell_id
            self.df.at[idx, 'lineage_id'] = lineage_id
            self.df.at[idx, 'generation'] = 0
            self.df.at[idx, 'parent_id'] = -1
            self.df.at[idx, 'parent_t'] = -1
            self.df.at[idx, 'division_event'] = False
            self.df.at[idx, 'birth_event'] = True
            self.df.at[idx, 'death_event'] = False

    def link_assignments(self, assignments: List[Dict[str, Any]]):
        """
        Consume ILP assignments for one or more frame transitions and
        update lineage info for children.

        assignments is a list, each element like:
            {
              'v': series_index,
              't': frame_index,
              'links': [
                   (parent_id, child_id)             # continuation
                   ...
              ],
              'divisions': [
                   (parent_id, child_id1, child_id2) # division
                   ...
              ],
              'births': [
                   child_id,                         # birth
                   ...
              ],
              'deaths': [
                   parent_id,                        # death
                   ...
              ]
            }

        This encodes the chosen mapping from frame t -> t+1 for series v.

        Rules:
        - Links and divisions both inherit lineage_id.
        - Links inherit cell_id. Divisions acquire a new cell_id.
        - Links inherit generation number. Divisions increment generation number.
        - Births get a new lineage_id and cell_id.
        - Deaths mark the parent as dead in frame t.
        """

        for A in assignments:
            v = A['v']
            t = A['t']
            links = A['links']
            divisions = A['divisions']
            births = A['births']
            deaths = A['deaths']

            for pid, cid in links:
                # locate parent row(s)
                parent_rows = self.df.query("v == @v and t == @t and id == @pid")

                # there should be exactly one row per (v,t,id)
                parent_idx = parent_rows.index[0]
                parent_key = (v, t, pid)

                lineage_id = self.lineage_map[parent_key]
                parent_generation = int(self.df.loc[parent_idx, 'generation'])
                cell_id = int(self.df.loc[parent_idx, 'cell_id'])

                child_rows = self.df.query("v == @v and t == @t+1 and id == @cid")
                child_idx = child_rows.index[0]

                self.df.loc[child_idx, 'parent_id'] = pid
                self.df.loc[child_idx, 'parent_t'] = t
                self.df.loc[child_idx, 'cell_id'] = cell_id
                self.df.loc[child_idx, 'lineage_id'] = lineage_id
                self.df.loc[child_idx, 'generation'] = parent_generation
                self.df.loc[child_idx, 'division_event'] = False
                self.df.loc[child_idx, 'birth_event'] = False
                self.df.loc[child_idx, 'death_event'] = False

                # record mapping for future propagation
                child_key = (v, t + 1, cid)
                self.lineage_map[child_key] = lineage_id
            
            for pid, cid1, cid2 in divisions:
                parent_rows = self.df.query("v == @v and t == @t and id == @pid")
                parent_idx = parent_rows.index[0]
                parent_key = (v, t, pid)

                lineage_id = self.lineage_map[parent_key]
                parent_generation = int(self.df.loc[parent_idx, 'generation'])

                for cid in [cid1, cid2]:
                    child_rows = self.df.query("v == @v and t == @t+1 and id == @cid")
                    child_idx = child_rows.index[0]

                    self.df.loc[child_idx, 'parent_id'] = pid
                    self.df.loc[child_idx, 'parent_t'] = t
                    self.df.loc[child_idx, 'cell_id'] = next(self.cell_counter)
                    self.df.loc[child_idx, 'lineage_id'] = lineage_id
                    self.df.loc[child_idx, 'generation'] = parent_generation + 1
                    self.df.loc[child_idx, 'division_event'] = True
                    self.df.loc[child_idx, 'birth_event'] = False
                    self.df.loc[child_idx, 'death_event'] = False 

                    # mark division event for the parent
                    self.df.loc[parent_idx, 'division_event'] = True

                    # record mapping for future propagation
                    child_key = (v, t + 1, cid)
                    self.lineage_map[child_key] = lineage_id
            
            for cid in births:
                child_rows = self.df.query("v == @v and t == @t+1 and id == @cid")
                child_idx = child_rows.index[0]

                self.df.loc[child_idx, 'parent_id'] = -1
                self.df.loc[child_idx, 'parent_t'] = -1
                self.df.loc[child_idx, 'cell_id'] = next(self.cell_counter)
                self.df.loc[child_idx, 'lineage_id'] = next(self.lineage_counter)
                self.df.loc[child_idx, 'generation'] = 0
                self.df.loc[child_idx, 'division_event'] = False
                self.df.loc[child_idx, 'birth_event'] = True
                self.df.loc[child_idx, 'death_event'] = False

                # record mapping for future propagation
                child_key = (v, t + 1, cid)
                self.lineage_map[child_key] = self.df.loc[child_idx, 'lineage_id']

            for pid in deaths:
                parent_rows = self.df.query("v == @v and t == @t and id == @pid")
                parent_idx = parent_rows.index[0]

                self.df.loc[parent_idx, 'death_event'] = True

    def finalize_lineages(
        self, 
        bad_frame_lookup: Optional[Dict[Tuple[int, int], bool]] = None
    ):
        """
        Assign lineage IDs to any remaining unlinked cells.
        This covers:
        - births that ILP did not connect to a parent (new arrivals),
        - frames you haven't run assignments on yet.

        If bad_frame_lookup is provided, it will be used to drop bad frame rows.

        Any such cell becomes a new lineage root with generation=0.
        If a cell already has a lineage_id we leave it alone.
        """

        unlinked_mask = self.df['lineage_id'] < 0
        unlinked_idx = self.df.index[unlinked_mask].to_numpy()

        drop_idx = []
        for idx in unlinked_idx:
            row = self.df.loc[idx]
            if bad_frame_lookup is not None and bad_frame_lookup.get((row['v'], row['t']), False):
                drop_idx.append(idx)
                if bad_frame_lookup.get((row['v'], row['t'] - 1), True):
                    self.mark_dead(row['v'], row['t'] - 1)
                continue

            key = (int(row['v']), int(row['t']), int(row['id']))
            if key in self.lineage_map:
                # already known (somehow partially set but line not filled?)
                # should be a dead code block normally...
                self.df.at[idx, 'lineage_id'] = self.lineage_map[key]
                if self.df.at[idx, 'generation'] < 0:
                    self.df.at[idx, 'generation'] = 0
                if self.df.at[idx, 'parent_id'] < 0:
                    self.df.at[idx, 'parent_id'] = -1
                if self.df.at[idx, 'parent_t'] < 0:
                    self.df.at[idx, 'parent_t'] = -1
                if pd.isna(self.df.at[idx, 'division_event']):
                    self.df.at[idx, 'division_event'] = False
                continue
            
            cell_id = next(self.cell_counter)
            lineage_id = next(self.lineage_counter)
            self.lineage_map[key] = lineage_id
            self.df.at[idx, 'cell_id'] = cell_id
            self.df.at[idx, 'lineage_id'] = lineage_id
            self.df.at[idx, 'generation'] = 0
            self.df.at[idx, 'parent_id'] = -1
            self.df.at[idx, 'parent_t'] = -1
            self.df.at[idx, 'division_event'] = False

        if len(drop_idx) > 0:
            self.df = self.df.drop(drop_idx)
    
    def mark_dead(self, v: int, t: int):
        """
        Mark all cells in series v at time t as dead.
        """
        self.df.loc[
            (self.df['v'] == v) & 
            (self.df['t'] == t), 
            'death_event'
        ] = True

    def get_descendant_indicies(self, index: int) -> list[int]:
        """
        Return the indicies of the descendants of a given cell index.
        """
        df = self.df

        v, t, pid = df.loc[index, 'v'], df.loc[index, 't'], df.loc[index, 'id']
        to_visit = [(v, t, pid)]
        visited = set(to_visit)
        lineage_rows = set([index])

        # Iteratively find all descendants
        while to_visit:
            v, t, pid = to_visit.pop()
            # Children at the next frame that reference this parent
            children = df.query("v == @v and parent_t == @t and parent_id == @pid")
            for _, child in children.iterrows():
                key = (child.series, child.frame, child.component_idx)
                if key not in visited:
                    visited.add(key)
                    to_visit.append(key)
                    lineage_rows.add(child.name)

        return list(lineage_rows)

    def get_lineage_tree(self, lineage_id: int) -> pd.DataFrame:
        """
        Return the full lineage tree for a given lineage_id, including all descendants.
        Traverses parent-child links to collect every generation of cells derived from
        the specified lineage root.

        Returns a DataFrame sorted by time ('t') and cell id ('cell_id').
        """
        df = self.df
        sub = df[df['lineage_id'] == lineage_id].copy()
        if sub.empty:
            return sub

        # Find all root nodes (cells with no parent in this lineage)
        root_cells = sub[sub['parent_id'] < 0][['v', 't', 'id']].to_records(index=False).tolist()
        visited = set(root_cells)
        to_visit = list(root_cells)
        lineage_rows = set(sub.index)

        # Iteratively find all descendants
        while to_visit:
            v, t, pid = to_visit.pop()
            # Children at the next frame that reference this parent
            children = df.query("v == @v and parent_t == @t and parent_id == @pid")
            for _, child in children.iterrows():
                key = (child.v, child.t, child.id)
                if key not in visited:
                    visited.add(key)
                    to_visit.append(key)
                    lineage_rows.add(child.name)

        full_tree = df.loc[list(lineage_rows)].copy()
        full_tree.sort_values(['t', 'cell_id'], inplace=True)
        return full_tree

# Define Cost matrix computation

In [ ]:
def build_cost_matrices(
    parents: np.ndarray,
    p_u: np.ndarray,
    p_len: np.ndarray,
    p_area: np.ndarray,
    p_lane: np.ndarray,
    p_div: np.ndarray,
    children: np.ndarray,
    c_u: np.ndarray,
    c_len: np.ndarray,
    c_area: np.ndarray,
    c_lane: np.ndarray,
    w_pos: float = 1.0,
    w_len: float = 1.0,
    w_area: float = 1.0,
    max_dist: float = 1.0,
    max_rel_len: float = 1.5,
    max_rel_area: float = 1.5,
    lambda_div: float = 1.0,
    lambda_survival: float = 1.0,
    k_survival: float = 1.0,
    u_offset: callable = lambda u: np.zeros_like(u),
    cost_birth: float = 10.0,
    u_death: callable = lambda u: 10.0 * np.ones_like(u),
    allow_division: bool = True,
) -> Dict[str, object]:
    """
    Build per-frame association costs between cells in df_t (time t) and df_tp1 (time t+1).

    Returns dict with:
        'parents': parent_ids  (list[int])
        'children': child_ids  (list[int])
        'div_pairs': list of (parent_id, child_id1, child_id2)
        'C_single': 2D np.ndarray [n_parents, n_children] with np.inf for disallowed
        'C_double': dict[(parent_id, child1, child2)] = cost_div  (np.inf if disallowed)
        'C_birth': cost of birth
        'C_death': np.ndarray [n_parents] with cost of death for each parent

    Rules:
    - C_single encodes the cost of a single edge link between parent i -> child j in the lineage tree.
    - div_pairs / C_double encode the cost for double edge links between parent i -> (child j, child k) in the lineage tree.
    - C_birth is the cost of a birth event for any child in df_tp1.
    - C_death is the cost of a death event for each parent in df_t.
    - allow_divison toggles whether double edges are considered.
    - A single-frame refractory window for division events is enforced.
    - Hard pruning is applied (distance, lane mismatch, etc). Disallowed edges get np.inf.

    Notes on cost design:
    - We normalize distances and angles by characteristic scales so terms are scale-free.
    - Area and length errors are expressed as relative differences, so are intrinsically scale-free.
    - max_* thresholds are used for candidate pruning to prevent ILP blow-up.
    - lambda_div is a scalar penalizing division events (increase to make divisions more rare).
        - lambda_survival is multiplier penalizing survival events when centroid_u approaches 1.
        - k_survival is the power of centroid_u for the survival cost.
    - u_offset is a function of parent_u that returns a small offset added to distance.
        - Useful to generate a directional bias in tracking.
        - Acts like a static flow field pushing cells in a preferred direction.
    - cost_birth is the baseline cost of birth.
    - u_death is a function of parent_u that returns the death cost.
    """

    p_u_clip = np.maximum(0.0, p_u) # p_u < 0 iff the roi was not defined perfectly -> breaks some power functions
    f_u = p_u_clip ** k_survival # NOTE: I'm not sure this is necessary... but I had included it when tuning and was afraid to remove it later.
                                 #       It penalizes survival more strongly away from trap root, but really just confuses things with the
                                 #       death cost function already present. Regardless, it's here now. It may help order cells within the trap?
                                 #       ... But this can probably be absorbed into u_death for similar outcomes.
                                 #       To be clear, those are not the same thing exactly. u_death computes the cost for slack variables which
                                 #       represent removal in the ILP, while this term increases the cost of C_single and C_double directly (
                                 #       the cost of producing single and double edges in the lineage graph).
    p_u_offset = u_offset(p_u_clip)

    nP = parents.size
    nC = children.size

    # characteristic scales for normalization
    # avoid zero division with small eps
    eps = 1e-9
    sigma_pos = max_dist if max_dist > 0 else 1.0

    # Build single-link cost matrix
    C_link = np.full((nP, nC), np.inf, dtype=float)

    for i in range(nP):
        dist = np.abs(c_u - p_u[i] - p_u_offset[i])
        dlen = np.abs(c_len - p_len[i]) / (p_len[i] + eps)
        darea = np.abs(c_area - p_area[i]) / (p_area[i] + eps)
        same_lane = (c_lane == p_lane[i])

        # prune implausible
        plausible_mask = (
            (dist <= max_dist) &
            (dlen <= max_rel_len - 1) &
            (darea <= max_rel_area - 1) &
            same_lane
        )

        if not np.any(plausible_mask):
            continue

        # compute cost only for plausible children
        local_cost = (
            w_pos * np.power((dist[plausible_mask] / sigma_pos), 2) +
            w_len * np.power((dlen[plausible_mask]), 2) +
            w_area * np.power((darea[plausible_mask]), 2) + 
            lambda_survival * f_u[i] # See NOTE above about this term
        )

        C_link[i, plausible_mask] = local_cost

    # Division candidates
    div_pairs = []
    C_division = {}

    if allow_division and nC >= 2:
        # pre-filter child pairs by proximity and lane consistency to reduce blowup
        # We will consider unordered pairs j<k
        for i in range(nP):
            # single frame refractory period for division events
            if p_div[i]:
                continue

            # candidate children that are individually plausible successors
            dist_all = np.abs(c_u - p_u[i] - p_u_offset[i])
            same_lane = (c_lane == p_lane[i])

            base_mask = (
                (dist_all <= max_dist) &
                same_lane
            )

            cand_idx = np.where(base_mask)[0]
            if cand_idx.size < 2:
                continue

            # now evaluate all unordered pairs among these candidates
            for a_i, b_i in itertools.combinations(cand_idx, 2):
                # enforce child-child proximity (daughters close to each other)
                sib_dist = np.abs(c_u[a_i] - c_u[b_i])
                if sib_dist > max_dist * 2:
                    continue

                # area consistency: parent area ~= sum(child areas)
                area_err = np.abs(c_area[a_i] + c_area[b_i] - p_area[i]) / (p_area[i] + eps)
                if area_err > max_rel_area - 1:
                    continue

                # length consistency: parent length ~= sum(child lengths) * max_rel_len
                len_err = np.abs(c_len[a_i] + c_len[b_i] - p_len[i]) / (p_len[i] + eps)
                if len_err > max_rel_len - 1:
                    continue

                # cost for division edge
                # combine geometric consistency of both daughters to parent
                dpos_a = np.abs(c_u[a_i] - p_u[i] - p_u_offset[i]) / sigma_pos
                dpos_b = np.abs(c_u[b_i] - p_u[i] - p_u_offset[i]) / sigma_pos

                cost_pair = (
                    w_pos * np.power(dpos_a + dpos_b, 2) +
                    w_area * np.power(area_err, 2) +
                    w_len * np.power(len_err, 2) +
                    lambda_survival * f_u[i] + # See NOTE above about this term
                    lambda_div
                )

                pid = int(parents[i])
                cid_a = int(children[a_i])
                cid_b = int(children[b_i])

                key = (pid, cid_a, cid_b)
                div_pairs.append(key)
                C_division[key] = float(cost_pair)
    
    C_death = u_death(p_u)

    return {
        'parents': parents.tolist(),
        'children': children.tolist(),
        'div_pairs': div_pairs,
        'C_single': C_link,
        'C_double': C_division,
        'C_birth': cost_birth,
        'C_death': C_death
    }

# Define ILP for lineage reconstruction

In [ ]:
def solve_ilp_assignment(
    parents: List[int],
    children: List[int],
    div_pairs: List[Tuple[int, int, int]],
    C_single: np.ndarray,
    C_double: Dict[Tuple[int, int, int], float],
    C_birth: float,
    C_death: np.ndarray,
    big_M: float = 1e6
) -> Dict[str, object]:
    """
    Solve the parent→child / parent→(child,child) assignment ILP for one frame transition.

    parents: list of parent ids at time t
    children: list of child ids at time t+1
    div_pairs: list of (p, c1, c2) unordered child pairs from same parent if in C_double
    C_single[i,j]: cost of assigning parent[i] -> child[j], or np.inf if forbidden
    C_double[(p,c1,c2)]: cost of assigning parent[p] -> (child[c1], child[c2]), or np.inf if forbidden
    C_birth: cost of birth, or np.inf if forbidden
    C_death[i]: cost of death for parent i, or np.inf if forbidden

    Constraints:
    - Each child at t+1 must be explained by exactly one parent,
      either as a single continuation OR as one of a division pair.
    - Each parent at t can spawn:
        0 children,
        1 child (continuation),
        or 2 children (division).
      No parent can claim >2 children.

    Output:
        {
          'links': [
             (parent_id, [child_id, ...]),
             ...
          ]
        }
    """

    nP = len(parents)
    nC = len(children)

    # index maps to locate i,j quickly
    parent_index = {pid: i for i, pid in enumerate(parents)}
    child_index = {cid: j for j, cid in enumerate(children)}

    # ---------
    # Variables
    # ---------

    # x_ij = 1 if parent i -> child j (single continuation)
    x_vars = {}
    # We'll only create variables for finite-cost edges
    for i, pid in enumerate(parents):
        for j, cid in enumerate(children):
            cost_ij = C_single[i, j]
            if np.isfinite(cost_ij):
                var_name = f"x_{pid}_{cid}"
                x_vars[(i, j)] = pulp.LpVariable(var_name, lowBound=0, upBound=1, cat='Binary')

    # y_i_jk = 1 if parent i divides into children j and k
    # We'll canonicalize (j,k) ordering so j<k in child index
    y_vars = {}
    for (pid, c1, c2) in div_pairs:
        i = parent_index[pid]
        j = child_index[c1]
        k = child_index[c2]
        j0, k0 = sorted((j, k))
        var_name = f"y_{pid}_{children[j0]}_{children[k0]}"
        y_vars[(i, j0, k0)] = pulp.LpVariable(var_name, lowBound=0, upBound=1, cat='Binary')

    # b_j = 1 if child i is born
    b_vars = {}
    for j, cid in enumerate(children):
        var_name = f"b_{cid}"
        b_vars[j] = pulp.LpVariable(var_name, lowBound=0, upBound=1, cat='Binary')

    # d_i = 1 if parent i dies
    d_vars = {}
    for i, pid in enumerate(parents):
        var_name = f"d_{pid}"
        d_vars[i] = pulp.LpVariable(var_name, lowBound=0, upBound=1, cat='Binary')

    # ------------
    # ILP optimization
    # ------------
    prob = pulp.LpProblem("frame_assignment", pulp.LpMinimize)

    # Objective:
    # minimize  Σ_i Σ_j [ C_single(i,j) * x_ij ] +
    #           Σ_i Σ_{j<k} [ C_div(i,j,k) * y_i_jk ] +
    #           Σ_j [ cost_birth * b_j ] +
    #           Σ_i [ cost_death * d_i ]
    obj_terms = []
    for (i, j), var in x_vars.items():
        obj_terms.append(C_single[i, j] * var)

    for (i, j, k), var in y_vars.items():
        pid = parents[i]
        cid_j = children[j]
        cid_k = children[k]
        key = (pid, cid_j, cid_k)
        # C_double was stored with (pid,c1,c2) where c1,c2 are actual IDs;
        # but j,k are indices. We must match ordering.
        if key not in C_double:
            key = (pid, cid_k, cid_j)
        cost_div = C_double.get(key, big_M)
        obj_terms.append(cost_div * var)
    
    for i, var in b_vars.items():
        obj_terms.append(C_birth * var)

    for i, var in d_vars.items():
        obj_terms.append(C_death[i] * var)

    prob += pulp.lpSum(obj_terms)

    # ----------------------------------------------
    # Constraint: every child has exactly one origin
    # ----------------------------------------------
    # For every child j at t+1:
    #   sum_i x_ij  +  sum_i sum_{k != j} y_i_jk  +  sum_j b_j  == 1
    # where y_i_jk is active if j or k is in that division.
    for j, cid_j in enumerate(children):
        terms_cover = []

        # single-parent terms x_ij
        for (i, jj) in x_vars:
            if jj == j:
                terms_cover.append(x_vars[(i, j)])

        # division terms y_i_jk that include child j
        for (i, jj, kk), var in y_vars.items():
            if jj == j or kk == j:
                terms_cover.append(var)
        
        # birth terms b_j
        terms_cover.append(b_vars[j])

        prob += pulp.lpSum(terms_cover) == 1, f"child_{cid_j}_origin_exclusivity"

    # ---------------------------------------------
    # Constraint: every parent has exactly one fate
    # ---------------------------------------------
    # For every parent i at t:
    #   sum_i x_ij  +  sum_i y_i_jk  +  sum_i d_i  == 1
    for i, pid in enumerate(parents):
        singles = [x_vars[(i, j)] for j in range(len(children)) if (i, j) in x_vars]
        divs    = [y_vars[(ii, jj, kk)] for (ii, jj, kk) in y_vars if ii == i]
        deaths  = [d_vars[i]]

        prob += pulp.lpSum(singles + divs + deaths) == 1, f"parent_{pid}_fate_exclusivity"

    # ---------
    # Solve ILP
    # ---------
    prob.solve(pulp.PULP_CBC_CMD(msg=False))

    # --------------
    # Extract result
    # --------------
    links = []
    for (i, j), var in x_vars.items():
        if var.value() is not None and var.value() > 0.5:
            pid = parents[i]
            cid = children[j]
            links.append((pid, cid))

    divisions = []
    for (i, j, k), var in y_vars.items():
        if var.value() is not None and var.value() > 0.5:
            pid = parents[i]
            cid_j = children[j]
            cid_k = children[k]
            divisions.append((pid, cid_j, cid_k))
    
    births = []
    for j, var in b_vars.items():
        if var.value() is not None and var.value() > 0.5:
            cid = children[j]
            births.append(cid)
    
    deaths = []
    for i, var in d_vars.items():
        if var.value() is not None and var.value() > 0.5:
            pid = parents[i]
            deaths.append(pid)

    return {
        'links': links,
        'divisions': divisions,
        'births': births,
        'deaths': deaths,
        'status': pulp.LpStatus[prob.status],
        'objective': pulp.value(prob.objective),
    }

# Load statistics dataframe from segmentation notebook

In [ ]:
experiment_name = 'local_folder_for_results'
statistics_df = pd.read_csv(f'backup/tables/{experiment_name}/statistics.csv')
print(f"{statistics_df['v'].nunique()} series found in {experiment_name}.")
statistics_df.head()

In [ ]:
# create a diction of (series, frame) -> True if bad frame, False if good frame
# used to manage frames you don't want to track through (e.g. due to bad imaging quality)
bad_frame_lookup = {}

# Compute lineage tracks using ILP assignment

In [ ]:
# number of series (v) and frames (t)
unique_series = np.sort(statistics_df['v'].unique())
unique_frames = np.sort(statistics_df['t'].unique())
tracker = LineageTracker(statistics_df[
    statistics_df['lane'] != -1
])

In [ ]:
# iterate over all series and frames, building cost matrices and solving ILP for each transition
series_completed = set()
for v in tqdm(unique_series,
              desc="Tracking lineages",
              position=0,
              dynamic_ncols=True,
              unit="series"):
    # initialize lineages for this series
    tracker.initialize_lineages(v, unique_frames[0])
    reinitialize = False

    # iterate over consecutive frame pairs
    for t in tqdm(unique_frames[:-1],
                desc=f"Processing series {v}",
                dynamic_ncols=True,
                position=1,
                leave=True,
                unit='frame'):
        # Check QC results
        if bad_frame_lookup.get((v, t), False) or bad_frame_lookup.get((v, t+1), False):
            reinitialize = True
            continue
        
        if reinitialize:
            tracker.initialize_lineages(v, t)
            reinitialize = False

        df_t = tracker.df[(tracker.df['v'] == v) & (tracker.df['t'] == t)]
        df_tp1 = tracker.df[(tracker.df['v'] == v) & (tracker.df['t'] == t+1)]
        if df_t.empty:
            raise Exception(f"Empty frame {t} for series {v}")
        if df_tp1.empty:
            raise Exception(f"Empty frame {t+1} for series {v}")

        # build cost matrices for this frame transition
        cm = build_cost_matrices(
            parents=df_t['id'].to_numpy(),
            p_u=df_t['centroid_u'].to_numpy(),
            p_len=df_t['cell_length'].to_numpy(),
            p_area=df_t['cell_area'].to_numpy(),
            p_lane=df_t['lane'].to_numpy(),
            p_div=df_t['division_event'].to_numpy(),
            children=df_tp1['id'].to_numpy(),
            c_u=df_tp1['centroid_u'].to_numpy(),
            c_len=df_tp1['cell_length'].to_numpy(),
            c_area=df_tp1['cell_area'].to_numpy(),
            c_lane=df_tp1['lane'].to_numpy(),
            w_pos = 2.0,
            w_len = 1.0,
            w_area = 1.0,
            max_dist = 0.3,
            max_rel_len = 1.5,
            max_rel_area = 1.5,
            lambda_div = 0.05,
            lambda_survival = 2.0,
            k_survival = 3.0,
            u_offset = lambda u: 0.1 * u**1.1,
            cost_birth = 10,
            u_death = lambda u: 1 / (np.exp(10*(u - 0.8)) + 1)
        )
        
        # solve ILP for this frame transition
        ilp_result = solve_ilp_assignment(
            parents=cm['parents'],
            children=cm['children'],
            div_pairs=cm['div_pairs'],
            C_single=cm['C_single'],
            C_double=cm['C_double'],
            C_birth=cm['C_birth'],
            C_death=cm['C_death']
        )

        if ilp_result['status'] != 'Optimal':
            warnings.warn(f"Series {v}, frame {t}: ILP status: {ilp_result['status']}", RuntimeWarning)
        
        # convert ILP result to tracker.link_assignments(...) format
        assignments = [{
            'v': int(v),
            't': int(t),
            'links': ilp_result['links'],
            'divisions': ilp_result['divisions'],
            'births': ilp_result['births'],
            'deaths': ilp_result['deaths']
        }]
        tracker.link_assignments(assignments)
    series_completed.add(v)

# finalize the dataframe
tracker.finalize_lineages(bad_frame_lookup=bad_frame_lookup)

In [ ]:
# save the full lineage dataframe to csv
tracker.df.to_csv(f"tables/{experiment_name}/lineages_raw.csv", index=False)